After Stage 2:

Stage 3 — Evaluation + **open-set** calibration

No new training.
* Drop ArcFace classifier
* Use embeddings only

Inference
* Build prototypes: per ID optionally per view (right / left)
* Cosine similarity
* Threshold for unknown rejection

Metrics
* Recall@k, mAP (closed-set)
* ROC / TAR@FAR (open-set)
* Few-shot gallery (1/2/3 images)

--> **Input:** No logits but a Distance Matrix ()

This stage answers:
* Does this actually work for new jaguars?


Google AI:

In ReID, we use:
* mAP (Mean Average Precision): How robust is the model? (Area under the Precision-Recall curve).
* CMC (Cumulative Match Characteristic) / Rank-k: "Is the correct jaguar in the top 1, top 5, or top 10 matches?"

Optional stages (only if Stage 2 fails)
Stage 4 — View-aware refinement

Only if cross-view performance is bad.

Options (pick one):

View-aware sampling

Separate left/right ArcFace heads (shared backbone)

View-specific prototypes only (no retraining)

Do not start here.

In [ ]:
import torch
import numpy as np

def compute_distance_matrix(q_feats, g_feats):
    """
    Computes distance between every Query (q) and every Gallery (g).
    Input: Normalized Tensors [Num_Queries, Dim], [Num_Gallery, Dim]
    Output: [Num_Queries, Num_Gallery]
    """
    # Matrix Multiplication gives Cosine Similarity (if normalized)
    sim_mat = torch.mm(q_feats, g_feats.t())
    # Convert to distance (Lower is better for sorting)
    dist_mat = 1 - sim_mat
    return dist_mat.cpu().numpy()

def evaluate_reid(dist_mat, q_pids, g_pids, topk=[1, 5]):
    """
    Standard Re-ID Evaluation.
    
    Args:
        dist_mat: numpy array [num_q, num_g] (pairwise distances)
        q_pids: numpy array [num_q] (true IDs of queries)
        g_pids: numpy array [num_g] (true IDs of gallery)
    """
    num_q, num_g = dist_mat.shape
    
    # Sort the gallery indices for every query (Ascending distance)
    # indices[i] = [idx_of_closest_match, idx_of_2nd_closest, ...]
    indices = np.argsort(dist_mat, axis=1)
    
    # Create a boolean matrix of matches
    # matches[i, j] is True if the j-th sorted gallery image has the same ID as query i
    matches = (g_pids[indices] == q_pids[:, np.newaxis])

    # --- Compute CMC (Rank-K) ---
    # cumsum gives us [0, 0, 1, 1, 2...] (how many matches found so far)
    # We only care if we found AT LEAST one match
    cmc = matches.cumsum(axis=1)
    cmc[cmc > 1] = 1
    
    all_cmc = cmc.mean(axis=0) # Average across all queries
    
    cmc_results = {f"Rank-{k}": all_cmc[k-1] for k in topk}

    # --- Compute mAP ---
    # This is the tricky retrieval mAP
    num_rel = matches.sum(axis=1) # How many correct matches exist for each query?
    tmp_cmc = matches.cumsum(axis=1)
    
    # Precision at k = (matches so far) / k
    # We calculate precision at every rank position
    precisions = tmp_cmc / np.arange(1, num_g + 1)
    
    # We only sum precision at positions where a match ACTUALLY occurred
    av_precisions = (precisions * matches).sum(axis=1) / num_rel
    
    # Handle cases where no matches exist in gallery
    av_precisions[np.isnan(av_precisions)] = 0 
    
    mAP = np.mean(av_precisions)
    
    return mAP, cmc_results